In [14]:
# Import the libraries
import os
import asyncio
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

In [15]:
# Initializing the enrollment setup

load_dotenv()

# Guardrails to ensure keys are loaded before running async loops
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("CRITICAL: OPENAI_API_KEY is missing from your environment")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "VS-Code-Advanced-Pipeline"

# Primary high-performance model configuration
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Fallback model configuration in case of rate limits or service disruption
fallback_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

In [16]:
# 2 Data layer (Pydantic Schemas for structured output)

class TechnicalReview(BaseModel):
    """Structured criteria evaluating incoming engineering requests."""
    category: Literal["architecture", "debugging", "general_inquiry"] = Field(
        ..., description="The primary domain classification of the user inquiry."
    )
    complexity: Literal["low", "medium", "high"] = Field(
        ..., description="Calculated implementation complexity of the query."
    )
    suggested_tools: List[str] = Field(
        default=[], description="Frameworks or tools relevant to resolving this issue (e.g., Docker,..."
    )

# Bind structural constraints to our model with a graceful fallback
structured_analyzer = llm.with_structured_output(TechnicalReview).with_fallbacks(
    [fallback_llm.with_structured_output(TechnicalReview)]
)

In [17]:
# Prompt Component Management

analyze_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the developer inquiry and output data structured precisely to the TechnicalReview schema."),
    ("user", "{input}")
])

architecture_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Enterprise Solutions Architect. Design robust architectural blueprints detailing data flows, scaling mechanics, and design patterns. List tools to consider: {suggested_tools}"),
    ("user", "{input}")
])

general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior software engineer. Answer the developer's technical query with clear, actionable guidance."),
    ("user", "{input}")
])

In [18]:
# Orchestration Layer (dynamic rout and branching)

architecture_chain = architecture_prompt | llm | StrOutputParser()
general_chain = general_prompt | llm | StrOutputParser()

def route_request(execution_state: dict):
    """Dynamic router parsing runtime state metrics."""
    metadata: TechnicalReview = execution_state["analysis"]

    print(f"\n⚡ [Router Log] Routing detected path: {metadata.category.upper()}")
    print(f"   [Context Metrics] Complexity: {metadata.complexity.upper()} | Expected Stack: {metadata.suggested_tools}")

    # Extract structural arguments forward into the sub-chain prompt context
    payload = {
        "input": execution_state["input"],
        "suggested_tools": ", ".join(metadata.suggested_tools) if metadata.suggested_tools else "None specified"
    }

    if metadata.category == "architecture":
        return architecture_chain.with_config(run_name="ArchitectureBranch").ainvoke(payload)
    else:
        return general_chain.with_config(run_name="EngineeringBranch").ainvoke(payload)
    

    # Assemble the core master execution pipeline using RunnableParallel and LCEL
pipeline = (
    RunnableParallel(
        input=RunnablePassthrough(),
        analysis=analyze_prompt | structured_analyzer
    )
    | RunnableLambda(route_request)
)


In [ ]:
# 5 Asynchronous Desktop Execution Harness

async def main():
    test_queries = [
        "How do I scale an LLM ingestion pipeline to process 10,000 PDFs per minute concurrently?",
        "Write a python code snippet using LangChain to pull metadata from a vector database."
    ]

    print("🚀 Starting Production Pipeline Execution inside VS Code Environment...")

    for query in test_queries:
        print("\n" + "="*80)
        print(f"🖥️ USER QUERY: '{query}'")
        print("="*80)

        try:
            # Running as true async invocation across the pipeline network
            response = await pipeline.ainvoke({"input": query})
            print("\n🤖 GENERATED OUTPUT:\n")
            print(response)

        except Exception as err:
            print(f"\n❌ Pipeline Execution Exception: {str(err)}")

if __name__ == "__main__":
    # Safe desktop application event loop management
    try:
        # Detect if an event loop is already open (prevents Jupyter/IDE conflicts)
        loop = asyncio.get_running_loop()
        task = loop.create_task(main())
    except RuntimeError:
        # Standard execution loop initialization
        asyncio.run(main())
        

🚀 Starting Production Pipeline Execution inside VS Code Environment...

🖥️ USER QUERY: 'How do I scale an LLM ingestion pipeline to process 10,000 PDFs per minute concurrently?'


f:\GEN_AI\final submission\Project 3 Conversational Q&A Chatbot With Message History\.venv\Lib\site-packages\langsmith\client.py:652: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f4cd-7881-bcd1-726be2c62e71; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f52b-7de0-9102-b4edeaa665fd; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f52b-7de0-9102-b4f1933bdb03; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f52b-7de0-9102-b5067b8e618a; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f52c-7fe3-8911-346a92d86ee3; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f534-7c81-a454


⚡ [Router Log] Routing detected path: ARCHITECTURE
   [Context Metrics] Complexity: HIGH | Expected Stack: ['Apache Kafka', 'Apache Flink', 'Kubernetes', 'AWS Lambda', 'Google Cloud Functions']

🤖 GENERATED OUTPUT:

<coroutine object RunnableBindingBase.ainvoke at 0x0000024BF480FD40>

🖥️ USER QUERY: 'Write a python code snippet using LangChain to pull metadata from a vector database.'


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f535-7a20-8d29-08ec559c60ec; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-ffa1-7e81-9079-766b10e33c5f; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-ffa1-7e81-9079-766b10e33c5f; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f534-7c81-a454-58b5bab2e013; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f534-7c81-a454-58a745a05eca; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f52b-7de0-9102-b5067b8e618a; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-f52b-7de0-9102-b4edeaa665fd; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id=019efa79-ffa3-7fa3-ba8b-dfde28ff5122; trace=019efa79-f4cd-7881-bcd1-726be2c62e71,id


⚡ [Router Log] Routing detected path: GENERAL_INQUIRY
   [Context Metrics] Complexity: MEDIUM | Expected Stack: ['LangChain', 'Vector Database SDK']

🤖 GENERATED OUTPUT:

<coroutine object RunnableBindingBase.ainvoke at 0x0000024BF495CB40>


C:\Users\User\AppData\Local\Temp\ipykernel_24280\1258251773.py:18: RuntimeWarning: coroutine 'RunnableBindingBase.ainvoke' was never awaited
  response = await pipeline.ainvoke({"input": query})
C:\Python314\Lib\asyncio\events.py:94: RuntimeWarning: coroutine 'RunnableBindingBase.ainvoke' was never awaited
  self._context.run(self._callback, *self._args)
Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019efa79-ffa8-76a1-a4dc-33ad30130ff6,id=019efa79-ffac-7210-9c17-e29af01a5b50; trace=019efa79-ffa8-76a1-a4dc-33ad30130ff6,id=019efa7a-0578-7503-a650-1f7514ad5d47; trace=019efa79-ffa8-76a1-a4dc-33ad30130ff6,id=019efa7a-0578-7503-a650-1f7514ad5d47; trace=019efa79-ffa8-76a1-a4dc-33ad30130ff6,id=019efa79-ffac-7210-9c17-e28bdfaf5899; trace=019efa79-ffa8-76a1